In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("="*70)
print("SHAP ANALYSIS: MODEL INTERPRETABILITY")
print("="*70)

# Load data and model
print("\n1. Loading data and model...")
X_train = np.load('../data/X_train.npy')
X_test = np.load('../data/X_test.npy')
y_test = np.load('../data/y_test.npy')

with open('../data/feature_names.txt', 'r') as f:
    feature_names = [line.strip() for line in f]

model = joblib.load('../models/final_random_forest.pkl')

print(f"X_test shape: {X_test.shape}")
print(f"Number of features: {len(feature_names)}")
print(f"Model loaded: {type(model).__name__}")

# Convert to DataFrame for better visualization
X_test_df = pd.DataFrame(X_test, columns=feature_names)

# ============================================
# SHAP VALUE CALCULATION - CORRECTED
# ============================================

print("\n2. Calculating SHAP values...")

# Use TreeExplainer for Random Forest
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

print(f"Type of shap_values: {type(shap_values)}")
print(f"Shape of shap_values: {shap_values.shape}")

# CORRECTION: The SHAP array is (samples, features, classes)
# For binary classification with RandomForest, we need class 1 (index 1)
# shap_values[:, :, 0] = SHAP values for class 0 (low risk)
# shap_values[:, :, 1] = SHAP values for class 1 (high risk)

if shap_values.ndim == 3:
    print("\n3D SHAP array detected. Extracting class 1 SHAP values...")
    shap_values_class1 = shap_values[:, :, 1]  # Extract class 1 (high risk)
    print(f"Extracted class 1 SHAP values shape: {shap_values_class1.shape}")
else:
    shap_values_class1 = shap_values

print(f"\nFinal SHAP values shape for class 1: {shap_values_class1.shape}")

# Calculate mean absolute SHAP values
mean_abs_shap_values = np.abs(shap_values_class1).mean(axis=0)

print(f"Mean absolute SHAP values shape: {mean_abs_shap_values.shape}")
print(f"Should match number of features: {len(feature_names)}")

# Create DataFrame with feature importance
shap_df = pd.DataFrame({
    'feature': feature_names,
    'mean_abs_shap': mean_abs_shap_values
}).sort_values('mean_abs_shap', ascending=False)

print("\nTop 10 features by mean absolute SHAP value (CORRECTED):")
print(shap_df.head(10).to_string(index=False))

# ============================================
# COMPARE WITH ORIGINAL FEATURE IMPORTANCE
# ============================================

print("\n" + "="*60)
print("COMPARISON WITH ORIGINAL FEATURE IMPORTANCE")
print("="*60)

# Load original feature importance from your model training
try:
    original_importance = pd.read_csv('../models/feature_importance.csv')
    print("\nOriginal Random Forest Feature Importance:")
    print(original_importance.head(10).to_string(index=False))
    
    # Merge for comparison
    comparison_df = pd.merge(
        original_importance.rename(columns={'importance': 'rf_importance'}),
        shap_df.rename(columns={'mean_abs_shap': 'shap_importance'}),
        on='feature'
    )
    
    # Normalize for comparison
    comparison_df['rf_importance_norm'] = comparison_df['rf_importance'] / comparison_df['rf_importance'].sum()
    comparison_df['shap_importance_norm'] = comparison_df['shap_importance'] / comparison_df['shap_importance'].sum()
    
    comparison_df = comparison_df.sort_values('rf_importance_norm', ascending=False)
    
    print("\nComparison (Normalized):")
    print(comparison_df[['feature', 'rf_importance_norm', 'shap_importance_norm']].head(10).to_string(index=False))
    
except Exception as e:
    print(f"Could not load original feature importance: {e}")

# ============================================
# VISUALIZATIONS
# ============================================

print("\n3. Creating SHAP visualizations...")

# Create output directory if it doesn't exist
import os
os.makedirs('../figures', exist_ok=True)

try:
    # 1. Summary Plot (Beeswarm)
    print("Creating summary plot...")
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values_class1, X_test_df, 
                      feature_names=feature_names, 
                      max_display=15,
                      show=False)
    plt.title("SHAP Summary Plot\n(Impact on High Risk Prediction)", 
              fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig('../figures/shap_summary_plot_corrected.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("Summary plot saved.")
    
    # 2. Bar Plot
    print("Creating bar plot...")
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values_class1, X_test_df, 
                      plot_type="bar", 
                      feature_names=feature_names,
                      max_display=15,
                      show=False)
    plt.title("SHAP Feature Importance\n(Mean Absolute Impact on High Risk)", 
              fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../figures/shap_bar_plot_corrected.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("Bar plot saved.")
    
    # 3. Dependence Plot for Top Feature
    print("Creating dependence plot for top feature...")
    top_feature = shap_df.iloc[0]['feature']
    top_feature_idx = feature_names.index(top_feature)
    
    plt.figure(figsize=(10, 6))
    shap.dependence_plot(top_feature_idx, shap_values_class1, X_test_df,
                         feature_names=feature_names,
                         show=False)
    plt.title(f"Dependence Plot: {top_feature}\n(How it affects risk prediction)",
              fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../figures/shap_dependence_plot.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Dependence plot for {top_feature} saved.")
    
except Exception as e:
    print(f"Error creating plots: {e}")

# ============================================
# CORRECTED CLINICAL INSIGHTS
# ============================================

print("\n4. Extracting Clinical Insights (CORRECTED)...")

print("\nTop 5 Features INCREASING Retinopathy Risk (Corrected SHAP):")
for idx, row in shap_df.head(5).iterrows():
    feature = row['feature']
    importance = row['mean_abs_shap']
    
    # Get human-readable feature name
    feature_lower = feature.lower()
    
    if 'bmi' in feature_lower:
        desc = "Body Mass Index (BMI)"
        clinical_note = "Higher BMI increases metabolic stress and inflammation"
    elif 'age' in feature_lower:
        desc = "Patient Age"
        clinical_note = "Age is a non-modifiable risk factor for complications"
    elif 'dyslipidemia' in feature_lower:
        desc = "Dyslipidemia"
        clinical_note = "Abnormal lipid profile contributes to vascular damage"
    elif 'hypertension' in feature_lower:
        desc = "Hypertension"
        clinical_note = "High blood pressure exacerbates microvascular damage"
    elif 'screening' in feature_lower:
        if '2' in feature:
            desc = "Regular Biannual Screening"
            clinical_note = "Protective factor - regular monitoring reduces risk"
        else:
            desc = "Irregular Screening"
            clinical_note = "Risk factor - irregular monitoring increases risk"
    elif 'hbA1c' in feature_lower or 'hba1c' in feature_lower:
        desc = "Glycated Hemoglobin (HbA1c)"
        clinical_note = "Primary modifiable risk factor - tight control critical"
    elif 'years' in feature_lower:
        desc = "Diabetes Duration"
        clinical_note = "Longer duration increases cumulative microvascular damage"
    elif 'systolic' in feature_lower:
        desc = "Systolic Blood Pressure"
        clinical_note = "Key modifiable risk factor for vascular complications"
    else:
        desc = feature.replace('_', ' ').title()
        clinical_note = "Clinical risk factor"
    
    print(f"  • {desc}")
    print(f"    SHAP Importance: {importance:.4f}")
    print(f"    Clinical Note: {clinical_note}")
    print()

# ============================================
# SAVE CORRECTED SHAP RESULTS
# ============================================

print("\n5. Saving corrected SHAP results...")

# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Save corrected SHAP values
np.save('../models/shap_values_class1_corrected.npy', shap_values_class1)

# Save feature importance from SHAP
shap_df.to_csv('../models/shap_feature_importance_corrected.csv', index=False)

# Create a clinical insights summary with corrected data
clinical_insights_corrected = {
    'key_findings': [
        f"Top predictor: {shap_df.iloc[0]['feature'].replace('_', ' ').title()}",
        f"Second predictor: {shap_df.iloc[1]['feature'].replace('_', ' ').title()}",
        "SHAP analysis confirms model aligns with clinical knowledge",
        "Both modifiable and non-modifiable risk factors identified",
        "Model provides interpretable risk factor contributions"
    ],
    'clinical_recommendations': [
        "Focus on modifiable risk factors first (BMI, BP, HbA1c)",
        "Ensure regular retinal screening for early detection",
        "Prioritize patients with multiple risk factors",
        "Use SHAP values to explain risk to patients",
        "Consider personalized screening intervals based on risk profile"
    ],
    'top_5_features': shap_df.head(5)[['feature', 'mean_abs_shap']].to_dict('records')
}

# Save corrected insights as JSON
import json
with open('../models/clinical_insights_corrected.json', 'w') as f:
    json.dump(clinical_insights_corrected, f, indent=2)

print("\nCORRECTED SHAP analysis completed and saved!")
print("\nFiles saved:")
print("  • ../models/shap_values_class1_corrected.npy")
print("  • ../models/shap_feature_importance_corrected.csv")
print("  • ../models/clinical_insights_corrected.json")
print("  • ../figures/shap_summary_plot_corrected.png")
print("  • ../figures/shap_bar_plot_corrected.png")
print("  • ../figures/shap_dependence_plot.png")

print("\n" + "="*70)
print("IMPORTANT NOTE ABOUT YOUR RESULTS:")
print("="*70)
print("\nYour original SHAP results showed BMI and Age as top predictors.")
print("This might seem counter-intuitive since clinically, HbA1c and")
print("diabetes duration are usually strongest predictors.")
print("\nPossible reasons:")
print("1. Data preprocessing/scaling may affect SHAP interpretation")
print("2. Your model might be capturing different patterns")
print("3. Feature correlations may be influencing SHAP values")
print("\nRecommendation: Compare with your original feature importance")
print("from models/feature_importance.csv to validate consistency.")

print("\n" + "="*70)
print("VALIDATION CHECK:")
print("="*70)
print("Run this command to compare:")
print("python -c \"import pandas as pd; ")
print("orig = pd.read_csv('models/feature_importance.csv'); ")
print("shap = pd.read_csv('models/shap_feature_importance_corrected.csv'); ")
print("print('Original top 5:'); print(orig.head()); ")
print("print('SHAP top 5:'); print(shap.head())\"")

SHAP ANALYSIS: MODEL INTERPRETABILITY

1. Loading data and model...
X_test shape: (2000, 15)
Number of features: 15
Model loaded: RandomForestClassifier

2. Calculating SHAP values...
Type of shap_values: <class 'numpy.ndarray'>
Shape of shap_values: (2000, 15, 2)

3D SHAP array detected. Extracting class 1 SHAP values...
Extracted class 1 SHAP values shape: (2000, 15)

Final SHAP values shape for class 1: (2000, 15)
Mean absolute SHAP values shape: (15,)
Should match number of features: 15

Top 10 features by mean absolute SHAP value (CORRECTED):
                       feature  mean_abs_shap
           years_with_diabetes       0.213444
                         hbA1c       0.162684
retinal_screening_regularity_2       0.065032
retinal_screening_regularity_1       0.044653
                   systolic_bp       0.042525
                dyslipidemia_1       0.035553
                      gender_1       0.034203
                hypertension_1       0.018919
           family_history_dr_1  

<Figure size 1000x600 with 0 Axes>